# Full Instance-Level TTA Gate (F1-1) — Colab runner

**Tujuan:** menjawab pertanyaan terbuka paper (Section V.F caveat) — apakah *gate* berbasis
**varians asli** antar-20-enumerasi (std / IQR) mengungguli *proxy* murah `|p_solo - p_TTA|`?
Sekaligus **menyimpan 20 prediksi mentah per molekul** sehingga caveat "per-variant predictions
were not retained" bisa dihapus dari paper.

**INFERENCE-ONLY** — memakai checkpoint ChemBERTa yang SUDAH dilatih. Tidak ada training ulang.
Checkpoint ~4 GB (set penuh `chemberta_{dataset}_{seed}.pt`) **bisa langsung dipakai**.

**Langkah:** (1) Runtime → GPU, (2) taruh checkpoint di Google Drive, (3) Run All.

Mode jalan otomatis:
- Jika folder `outputs/predictions/` (hasil run lama) juga diunggah → angka solo/TTA **identik paper**.
- Jika hanya checkpoint → **self-contained**: solo/TTA/std dihitung ulang dari 20 varian (angka ≈ paper, reproduksi segar). Temuan (sinyal mana yang terbaik) sama.

In [ ]:
# 1. Cek GPU + mount Google Drive (tempat checkpoint diunggah)
import torch, os
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'TIDAK ADA -> set Runtime>Change runtime type>GPU')
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone repo publik + masuk foldernya
REPO = 'https://github.com/belvahector-ship-it/pharm_.git'
if not os.path.exists('/content/pharm_'):
    !git clone -q {REPO} /content/pharm_
%cd /content/pharm_
!git pull -q && git log --oneline -1

In [ ]:
# 3. Dependency — versi DIPIN ke environment run asli (torch bawaan Colab dipertahankan)
!pip install -q 'rdkit==2026.3.3' 'transformers==5.0.0' 'tokenizers==0.22.2' 'chemprop==2.2.4'
print('install selesai (versi dipin)')

In [ ]:
# 4. Salin checkpoint dari Drive -> outputs/checkpoints/
#    >>> EDIT baris ini ke folder tempat Anda menaruh file *.pt di Drive <<<
CKPT_DRIVE_DIR = '/content/drive/MyDrive/pharm_checkpoints'
import glob, shutil, os
os.makedirs('outputs/checkpoints', exist_ok=True)
pts = glob.glob(os.path.join(CKPT_DRIVE_DIR, '**', '*.pt'), recursive=True)
assert pts, f'Tak ada *.pt di {CKPT_DRIVE_DIR} — cek path folder Drive Anda.'
for p in pts:
    dst = os.path.join('outputs/checkpoints', os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)
found = sorted(os.path.basename(x) for x in glob.glob('outputs/checkpoints/chemberta_*.pt'))
print(f'{len(found)} checkpoint ChemBERTa siap:'); print('\n'.join(found))
# (opsional) jika Anda juga punya folder outputs/predictions/ lama di Drive, salin agar angka identik paper:
PRED_DRIVE_DIR = '/content/drive/MyDrive/pharm_predictions'   # kosongkan/biarkan bila tak ada
if os.path.isdir(PRED_DRIVE_DIR):
    os.makedirs('outputs/predictions', exist_ok=True)
    n=0
    for p in glob.glob(os.path.join(PRED_DRIVE_DIR, '**', '*.npy'), recursive=True):
        d=os.path.join('outputs/predictions', os.path.basename(p))
        if not os.path.exists(d): shutil.copy(p,d); n+=1
    print(f'{n} prediksi tersimpan disalin (mode faithful).')
else:
    print('Tak ada folder prediksi lama -> mode self-contained (regen dari checkpoint).')

In [ ]:
# 5. Jalankan eksperimen F1-1 (semua dataset, 10 seed, backbone base). ~10-20 menit di T4.
#    --save_raw menyimpan 20 varian mentah (menghapus caveat 'not retained').
#    Hanya kombinasi yang checkpoint-nya ada yang diproses; sisanya di-skip otomatis.
!python scripts/14_full_instance_gate.py --backbones chemberta --save_raw 2>&1 | grep -vE 'Explicit valence|Loading weights|HF_TOKEN|RuntimeWarning|Degrees of freedom'

In [ ]:
# 6. Lihat hasil ringkas: apakah std / IQR (varians asli) mengalahkan proxy disagreement?
import pandas as pd
s = pd.read_csv('outputs/results/full_gate/full_instance_gate_summary.csv')
pd.set_option('display.width', 200, 'display.max_columns', 50)
print(s.to_string(index=False))
print('\n--- interpretasi ---')
print('Bandingkan kolom gate_disagree vs gate_std vs gate_iqr per dataset.')
print('Jika proxy (disagree) >= std & iqr -> temuan paper KONFIRMASI: sinyal murah cukup.')
print('Jika std/iqr > disagree signifikan -> varians asli menang -> update Table V + klaim.')

In [ ]:
# 7. Simpan hasil kembali ke Drive (CSV + report + varian mentah)
OUT_DRIVE = '/content/drive/MyDrive/pharm_full_gate_results'
os.makedirs(OUT_DRIVE, exist_ok=True)
for f in glob.glob('outputs/results/full_gate/*'):
    shutil.copy(f, OUT_DRIVE)
raws = glob.glob('outputs/predictions/chemberta_tta_raw_*.npz')
os.makedirs(os.path.join(OUT_DRIVE, 'raw_variants'), exist_ok=True)
for f in raws:
    shutil.copy(f, os.path.join(OUT_DRIVE, 'raw_variants'))
print(f'Hasil disalin ke {OUT_DRIVE} (+{len(raws)} berkas varian mentah).')
print('Kirim balik full_instance_gate_summary.csv untuk saya masukkan ke paper bila perlu.')